# Exploratory Data Analysis — Home Credit Default Risk

This notebook explores the `application_train.csv` file from the Home Credit Default Risk dataset: target class imbalance, missing values, key feature distributions by default status, categorical default rates, and feature correlations. Findings here inform feature engineering and modeling in later notebooks.

Generated charts are saved to `../data/processed/`.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Load Data

In [ ]:
df = pd.read_csv("../data/raw/application_train.csv")
df.head()


## 3. Basic Info

In [ ]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("\nData types:")
print(df.dtypes.value_counts())
df.info(verbose=False, memory_usage="deep")


## 4. Target Variable — Class Imbalance

In [ ]:
target_counts = df["TARGET"].value_counts().sort_index()
target_pct = df["TARGET"].value_counts(normalize=True).sort_index() * 100

print("Target distribution:")
print(target_counts)
print(f"\nDefault rate: {target_pct[1]:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0], palette=["#4C72B0", "#C44E52"])
axes[0].set_xticklabels(["No Default (0)", "Default (1)"])
axes[0].set_title("Target Class Counts")
axes[0].set_ylabel("Count")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom")

axes[1].pie(
    target_counts.values,
    labels=["No Default", "Default"],
    autopct="%1.1f%%",
    colors=["#4C72B0", "#C44E52"],
    startangle=90,
)
axes[1].set_title("Target Class Proportion")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/target_distribution.png", bbox_inches="tight")
plt.show()


## 5. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)

print(f"{len(missing_df)} of {df.shape[1]} columns have missing values")
missing_df.head(20)


In [ ]:
top_missing = missing_df.head(30).sort_values("missing_pct")

fig, ax = plt.subplots(figsize=(10, 10))
ax.barh(top_missing.index, top_missing["missing_pct"], color="#DD8452")
ax.set_xlabel("Missing (%)")
ax.set_title("Top 30 Columns by Missing Value Percentage")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/missing_values.png", bbox_inches="tight")
plt.show()


## 6. Key Feature Distributions — Default vs No Default

In [ ]:
numeric_features = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_ANNUITY", "DAYS_BIRTH"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    plot_df = df[[col, "TARGET"]].copy()
    if col == "AMT_INCOME_TOTAL":
        # clip extreme outliers for a readable plot
        upper = plot_df[col].quantile(0.99)
        plot_df = plot_df[plot_df[col] <= upper]

    sns.kdeplot(data=plot_df[plot_df["TARGET"] == 0], x=col, ax=axes[i],
                label="No Default", fill=True, alpha=0.4, color="#4C72B0")
    sns.kdeplot(data=plot_df[plot_df["TARGET"] == 1], x=col, ax=axes[i],
                label="Default", fill=True, alpha=0.4, color="#C44E52")
    axes[i].set_title(f"{col} Distribution by Target")
    axes[i].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_distributions.png", bbox_inches="tight")
plt.show()


In [ ]:
ext_source_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(ext_source_cols):
    sns.kdeplot(data=df[df["TARGET"] == 0], x=col, ax=axes[i],
                label="No Default", fill=True, alpha=0.4, color="#4C72B0")
    sns.kdeplot(data=df[df["TARGET"] == 1], x=col, ax=axes[i],
                label="Default", fill=True, alpha=0.4, color="#C44E52")
    axes[i].set_title(f"{col} by Target")
    axes[i].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ext_source_distributions.png", bbox_inches="tight")
plt.show()


## 7. EXT_SOURCE Boxplots by Target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(ext_source_cols):
    sns.boxplot(data=df, x="TARGET", y=col, ax=axes[i], palette=["#4C72B0", "#C44E52"])
    axes[i].set_xticklabels(["No Default", "Default"])
    axes[i].set_title(f"{col} by Target")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ext_source_boxplots.png", bbox_inches="tight")
plt.show()


## 8. Categorical Features — Default Rates

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print(f"{len(categorical_cols)} categorical columns: {categorical_cols}")


In [ ]:
key_cats = [c for c in ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE",
                         "NAME_FAMILY_STATUS", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE"]
            if c in categorical_cols]

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(key_cats):
    rates = df.groupby(col)["TARGET"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rates.values, y=rates.index, ax=axes[i], color="#C44E52")
    axes[i].set_xlabel("Default Rate (%)")
    axes[i].set_title(f"Default Rate by {col}")
    axes[i].axvline(df["TARGET"].mean() * 100, color="black", linestyle="--",
                     linewidth=1, label="Overall avg")
    axes[i].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/categorical_default_rates.png", bbox_inches="tight")
plt.show()


## 9. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
correlations = numeric_df.corr()["TARGET"].abs().sort_values(ascending=False)
top_features = correlations.index[1:21].tolist()  # exclude TARGET itself

corr_matrix = df[top_features + ["TARGET"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0, ax=ax,
            square=True, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Heatmap — Top 20 Features Correlated with TARGET")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap.png", bbox_inches="tight")
plt.show()


## 10. Summary of Key Findings

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

default_rate = df["TARGET"].mean() * 100
print(f"1. Class imbalance: {default_rate:.2f}% of applicants defaulted "
      f"({df['TARGET'].sum():,} of {len(df):,}) — a highly imbalanced target.")

n_missing_cols = (df.isnull().mean() > 0).sum()
n_high_missing = (df.isnull().mean() > 0.5).sum()
print(f"2. Missing data: {n_missing_cols} columns have missing values; "
      f"{n_high_missing} columns are missing more than 50% of values.")

top_corr = correlations.index[1:6].tolist()
print(f"3. Features most correlated with TARGET: {top_corr}")

print("4. EXT_SOURCE_1/2/3 show visibly lower scores for defaulters, "
      "confirming their value as strong predictive features.")

print("5. Categorical variables such as NAME_EDUCATION_TYPE and "
      "NAME_INCOME_TYPE show meaningful spread in default rate across categories.")

print("\nCharts saved to:", OUTPUT_DIR)
